In [1]:
# --- Data & tokenizer --------------------------------------------------------
import itertools, random, re, pandas as pd
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.decoders import WordPiece
from transformers import PreTrainedTokenizerFast
import torch, numpy as np

DET = {'fr':'le', 'nl':'de'}
DET_PL = {'fr':'les', 'nl':'de'}                    # same article in nl

NOUNS = {                                           # singular -> plural
    'fr': {'chien':'chiens','loup':'loups','chat':'chats','tigre':'tigres',
           'singe':'singes','cheval':'chevaux','souris':'souris',
           'éléphant':'éléphants','renard':'renards','oiseau':'oiseaux'},
    'nl': {'hond':'honden','leeuw':'leeuwen','kat':'katten','tijger':'tijgers',
           'aap':'apen','paard':'paarden','muis':'muizen',
           'olifant':'olifanten','vos':'vossen','vogel':'vogels'}
}

VERBS = {
    'fr':[('mange','mangé'),('voit','vu'),('aime','aimé'),('cherche','cherché'),
          ('aide','aidé'),('suit','suivi'),('frappe','frappé'),('porte','porté'),
          ('embrasse','embrassé'),('chasse','chassé')],
    'nl':[('eet','gegeten'),('ziet','gezien'),('mag','gemogen'),('zoekt','gezocht'),
          ('helpt','geholpen'),('volgt','gevolgd'),('slaat','geslagen'),
          ('draagt','gedragen'),('zoent','gezoend'),('jaagt','gejaagd')]
}

def make_pair(lang, subj, obj, pres, part, plural=False):
    det, det_pl = DET[lang], DET_PL[lang]
    s, o = (NOUNS[lang][subj] if plural else subj,
            NOUNS[lang][obj]  if plural else obj)
    d_s, d_o = (det_pl if plural else det,)*2

    inp = f"{d_s} {s} {pres} {d_o} {o}"
    if lang=='fr':
        tgt = f"{d_s} {s} a {part} {d_o} {o}"
    else:
        tgt = f"{d_s} {s} heeft {d_o} {o} {part}"
    return inp, tgt

rows=[]
rng = np.random.default_rng(0)
for lang in ('fr','nl'):
    nouns = list(NOUNS[lang])
    for subj,obj,(pres,part) in itertools.product(nouns,nouns,VERBS[lang]):
        if subj==obj: continue
        plural = rng.random() < 0.33
        inp,tgt = make_pair(lang,subj,obj,pres,part,plural)
        rows.append({'input':inp,'target':tgt,'lang':lang})
random.shuffle(rows)

full_df = pd.DataFrame(rows)
train_df = full_df.sample(frac=0.8,random_state=0)
test_df  = full_df.drop(train_df.index)

print("train / test", len(train_df), len(test_df))

# ---- tokenizer (same as before) --------------------------------------------
def tok(s): return re.findall(r"\w+|[^\s\w]", s)
special=['<pad>','<sos>','<eos>','<unk>']
vocab = set(it for col in ('input','target') for s in train_df[col] for it in tok(s))
vocab = special + sorted(vocab - set(special))
stoi  = {t:i for i,t in enumerate(vocab)}

tok_obj = Tokenizer(WordLevel(vocab=stoi,unk_token="<unk>"))
tok_obj.pre_tokenizer = Whitespace(); tok_obj.decoder = WordPiece()
tokenizer = PreTrainedTokenizerFast(tokenizer_object=tok_obj,
    bos_token="<sos>",eos_token="<eos>",unk_token="<unk>",pad_token="<pad>")

class HFDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.enc = tokenizer(df.input.tolist(), text_target=df.target.tolist(),
                             padding='max_length',truncation=True,max_length=20)
    def __len__(self): return len(self.enc['input_ids'])
    def __getitem__(self,i): return {k:torch.tensor(v[i]) for k,v in self.enc.items()}


/n/home06/drooryck/circuits_languages_2/venv39/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


train / test 1440 360


In [ ]:
!hostname

In [2]:
### TESTING CELL

import torch

# 1) GPU diagnostics
print("CUDA available? ", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Current device index:", torch.cuda.current_device())
    print("Device name:", torch.cuda.get_device_name(torch.cuda.current_device()))

# 2) Vocab size
try:
    vocab_size = tokenizer.vocab_size
except NameError:
    vocab_size = len(stoi)
print("Vocabulary size:", vocab_size)

# 3) Sample tokenization + IDs for a few test sentences
samples = test_df['input'].tolist()[:3]
print("\nSample tokenization and IDs:")
for s in samples:
    enc = tokenizer(s, padding='max_length', truncation=True, max_length=20)
    print(f"Input: {s}")
    print(" Tokens:", tokenizer.tokenize(s))
    print(" Input IDs:", enc['input_ids'])
    print(" Attention mask:", enc['attention_mask'])
    print()

# 4) (Optional) run model.generate if a model is in scope
if 'model' in globals():
    model_device = next(model.parameters()).device
    print(f"Model is on device: {model_device}")
    print("\nSample generations:")
    for s in samples:
        inputs = tokenizer(s, return_tensors='pt', padding=True).to(model_device)
        out = model.generate(**inputs, max_length=20)
        print(f"{s}  ->  {tokenizer.decode(out[0], skip_special_tokens=True)}")



CUDA available?  True
CUDA device count: 1
Current device index: 0
Device name: NVIDIA A100-SXM4-40GB MIG 3g.20gb
Vocabulary size: 88

Sample tokenization and IDs:
Input: de honden draagt de vogels
 Tokens: ['de', 'honden', 'draagt', 'de', 'vogels']
 Input IDs: [21, 41, 22, 21, 77, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
 Attention mask: [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Input: de paarden mag de honden
 Tokens: ['de', 'paarden', 'mag', 'de', 'honden']
 Input IDs: [21, 61, 51, 21, 41, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
 Attention mask: [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Input: les souris porte les chats
 Tokens: ['les', 'souris', 'porte', 'les', 'chats']
 Input IDs: [48, 69, 62, 48, 14, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
 Attention mask: [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]



In [4]:
# Cell 2: Train decoder-only Transformer at p=[0,0.5,1.0], with W&B off

from transformers import GPT2Config, GPT2LMHeadModel, Trainer, TrainingArguments

config = GPT2Config(
    vocab_size=tokenizer.vocab_size,
    n_embd=512,
    n_layer=4,
    n_head=8,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id
)

def make_train_ds(p):
    total=len(train_df)
    n_fr=int(total*p); n_nl=total-n_fr
    df_fr=train_df[train_df.lang=='fr'].sample(n_fr,replace=True,random_state=0)
    df_nl=train_df[train_df.lang=='nl'].sample(n_nl,replace=True,random_state=0)
    return HFDataset(pd.concat([df_fr,df_nl]).sample(frac=1,random_state=0))

for p in [0.0,0.5,1.0]:
    print(f"\n### {int(p*100)}% French ###")
    train_ds = make_train_ds(p)
    model = GPT2LMHeadModel(config)

    args = TrainingArguments(
        output_dir=f'out_p{int(p*100)}',
        max_steps=30000,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        learning_rate=1e-4,
        weight_decay=0.01,
        logging_steps=500,
        save_steps=1000,
        eval_steps=1000,
        eval_strategy='steps',
        report_to=[]            # ← disable all logging (no wandb, no tensorboard)
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=HFDataset(test_df)
    )

    trainer.train()

    print(f"--- Samples p={p} ---")
    for inp in test_df['input'].tolist()[:5]:
        inputs = tokenizer(inp, return_tensors='pt').to(trainer.model.device)
        out_ids = model.generate(**inputs, max_length=20)
        gen = tokenizer.decode(out_ids[0], skip_special_tokens=True)
        tgt = test_df.loc[test_df.input==inp,'target'].iloc[0]
        print(f"{inp} → {gen} | {tgt}")



### 0% French ###


Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
1000,0.252300,1.606845
2000,0.234400,1.753946
3000,0.224700,1.841894
4000,0.220600,1.896357
5000,0.218100,2.049363
6000,0.216400,2.050187
7000,0.215300,2.089329
8000,0.214300,2.210498
9000,0.213800,2.254337
10000,0.212500,2.314465


--- Samples p=0.0 ---
de honden draagt de vogels → de honden draagt de vogels gedragen | de honden heeft de vogels gedragen
de paarden mag de honden → de paarden mag de honden gemogen | de paarden heeft de honden gemogen
les souris porte les chats → les souris porte les chats de | les souris a porté les chats
de aap zoekt de hond → de aap zoekt de hond gezocht | de aap heeft de hond gezocht
le singe mange le oiseau → le singe mange le oiseau de | le singe a mangé le oiseau

### 50% French ###


Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Step,Training Loss,Validation Loss
1000,0.188600,0.221219
2000,0.178300,0.241866
3000,0.171100,0.257235
4000,0.167700,0.268045


KeyboardInterrupt: 

## bigger exp:

In [ ]:
#!/usr/bin/env python
# ---------------------------------------------------------------------
# Generates the Dutch/French present→past corpus and an 80/20 split.
# Exhaustively includes both singular and plural variants for each combination.
# ---------------------------------------------------------------------
import itertools, re, json, pandas as pd
from pathlib import Path

OUT_DIR = Path("data"); OUT_DIR.mkdir(exist_ok=True)

DET     = {'fr':'le',  'nl':'de'}
DET_PL  = {'fr':'les', 'nl':'de'}               # same article in nl
NOUNS = {
    'fr': {
        'chien': 'chiens', 'loup': 'loups', 'chat': 'chats', 'tigre': 'tigres',
        'singe': 'singes', 'cheval': 'chevaux', 'souris': 'souris',
        'éléphant': 'éléphants', 'renard': 'renards', 'oiseau': 'oiseaux',
        'enfant': 'enfants', 'docteur': 'docteurs', 'fille': 'filles',
        'garçon': 'garçons', 'femme': 'femmes', 'homme': 'hommes',
        'professeur': 'professeurs', 'ami': 'amis', 'ennemi': 'ennemis'
    },
    'nl': {
        'hond': 'honden', 'leeuw': 'leeuwen', 'kat': 'katten', 'tijger': 'tijgers',
        'aap': 'apen', 'paard': 'paarden', 'muis': 'muizen',
        'olifant': 'olifanten', 'vos': 'vossen', 'vogel': 'vogels',
        'kind': 'kinderen', 'dokter': 'dokters', 'meisje': 'meisjes',
        'jongen': 'jongens', 'vrouw': 'vrouwen', 'man': 'mannen',
        'leraar': 'leraars', 'vriend': 'vrienden', 'vijand': 'vijanden'
    }
}
VERBS = {
    'fr': [
        ('mange', 'mangé'), ('voit', 'vu'), ('aime', 'aimé'), ('cherche', 'cherché'),
        ('aide', 'aidé'), ('suit', 'suivi'), ('frappe', 'frappé'), ('porte', 'porté'),
        ('embrasse', 'embrassé'), ('chasse', 'chassé'),
        ('observe', 'observé'), ('soigne', 'soigné'), ('poursuit', 'poursuivi'),
        ('attaque', 'attaqué'), ('sauve', 'sauvé'), ('réconforte', 'réconforté'),
        ('repousse', 'repoussé'), ('regarde', 'regardé'), ('ignore', 'ignoré')
    ],
    'nl': [
        ('eet', 'gegeten'), ('ziet', 'gezien'), ('mag', 'gemogen'), ('zoekt', 'gezocht'),
        ('helpt', 'geholpen'), ('volgt', 'gevolgd'), ('slaat', 'geslagen'),
        ('draagt', 'gedragen'), ('zoent', 'gezoend'), ('jaagt', 'gejaagd'),
        ('observeert', 'geobserveerd'), ('verzorgt', 'verzorgd'), ('achtervolgt', 'achtervolgd'),
        ('valt', 'aangevallen'), ('redt', 'gered'), ('troost', 'getroost'),
        ('duwt', 'geduwd'), ('bekijkt', 'bekeken'), ('negeert', 'genegeerd')
    ]
}
def make_pair(lang, subj, obj, pres, part, plural):
    det, det_pl = DET[lang], DET_PL[lang]
    s, o = (NOUNS[lang][subj] if plural else subj,
            NOUNS[lang][obj]  if plural else obj)
    d_s, d_o = (det_pl if plural else det,)*2
    inp = f"{d_s} {s} {pres} {d_o} {o}"
    tgt = (f"{d_s} {s} a {part} {d_o} {o}"
           if lang=='fr'
           else f"{d_s} {s} heeft {d_o} {o} {part}")
    return inp, tgt

rows=[]
for lang in ('fr','nl'):
    nouns = list(NOUNS[lang])
    for subj, obj, (pres, part) in itertools.product(nouns, nouns, VERBS[lang]):
        if subj == obj:
            continue
        for plural in (False, True):  # generate both versions
            inp, tgt = make_pair(lang, subj, obj, pres, part, plural)
            rows.append({
                'input': inp,
                'target': tgt,
                'lang': lang,
                'plural': plural
            })

print(f"Generated {len(rows)} examples (with plural + singular for each pair).")
df = pd.DataFrame(rows)
train_df = df.sample(frac=0.8, random_state=0)
test_df  = df.drop(train_df.index)

train_df.to_csv(OUT_DIR/"train.csv", index=False)
test_df.to_csv (OUT_DIR/"test.csv", index=False)
print("Saved", len(train_df), "train +", len(test_df), "test lines")

# ---------- store vocab for reproducibility ------------------------------
def tok(s): return re.findall(r"\w+|[^\s\w]", s)
special = ['<pad>','<sos>','<eos>','<unk>']
vocab = special + sorted({t for s in df.input.tolist()+df.target.tolist() for t in tok(s)} - set(special))
json.dump(vocab, open(OUT_DIR/"vocab.json","w"), ensure_ascii=False, indent=2)
print("Vocab size:", len(vocab))``


Generated 3600 examples (with plural + singular for each pair).
Saved 2880 train + 720 test lines
Vocab size: 88


## RUN A COUPLE EXAMPLES

In [2]:
from pathlib import Path
import torch
import pandas as pd
from transformers import GPT2LMHeadModel, PreTrainedTokenizerFast
import json

# 1. Set up paths and parameters
p = 0.0  # Set to the proportion value you used during training (e.g., 0.0, 0.5, 1.0)
model_path = Path(f"weights/out_p{int(p * 100)}/final")
data_dir = Path("data")
results_file = Path(f"examples_p{int(p * 100)}.txt")

# 2. Load tokenizer and vocabulary
vocab = json.load(open(data_dir/"vocab.json"))
stoi = {t:i for i,t in enumerate(vocab)}
from tokenizers import Tokenizer; from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Whitespace; from tokenizers.decoders import WordPiece
tok_obj = Tokenizer(WordLevel(vocab=stoi, unk_token="<unk>"))
tok_obj.pre_tokenizer = Whitespace(); tok_obj.decoder = WordPiece()
tokenizer = PreTrainedTokenizerFast(tokenizer_object=tok_obj,
    bos_token="<sos>", eos_token="<eos>", unk_token="<unk>", pad_token="<pad>")

# 3. Load the trained model
model = GPT2LMHeadModel.from_pretrained(model_path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# 4. Load test data
test_df = pd.read_csv(data_dir/"test.csv")

# 5. Generate examples
num_examples = 20
examples = []

print(f"Generating examples with model trained on {p*100}% French...")
for _, row in test_df.sample(num_examples).iterrows():
    input_text = row['input']
    target_text = row['target']
    lang = row['lang']
    
    # Tokenize and move to device
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(device)
    
    # Generate output
    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_length=20,
            do_sample=False,
            num_return_sequences=1
        )
    
    # Decode
    predicted_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    
    # Store results
    examples.append({
        "language": lang,
        "input": input_text,
        "target": target_text,
        "prediction": predicted_text,
        "correct": predicted_text.strip() == target_text.strip()
    })
    
    # Print progress
    print(f"{lang}: {input_text} → {predicted_text} | Target: {target_text}")

# 6. Save results to file
with open(results_file, 'w') as f:
    f.write(f"Results for model trained with {p*100}% French\n")
    f.write("="*80 + "\n\n")
    
    # Calculate accuracy
    correct = sum(1 for ex in examples if ex["correct"])
    f.write(f"Accuracy: {correct}/{len(examples)} = {correct/len(examples):.2%}\n\n")
    
    # Group by language
    for lang in ['fr', 'nl']:
        lang_examples = [ex for ex in examples if ex['language'] == lang]
        if lang_examples:
            correct_lang = sum(1 for ex in lang_examples if ex["correct"])
            f.write(f"{lang.upper()} Accuracy: {correct_lang}/{len(lang_examples)} = {correct_lang/len(lang_examples):.2%}\n\n")
    
    # Examples
    for i, ex in enumerate(examples):
        f.write(f"Example {i+1} ({ex['language']}):\n")
        f.write(f"Input:      {ex['input']}\n")
        f.write(f"Target:     {ex['target']}\n")
        f.write(f"Prediction: {ex['prediction']}\n")
        f.write(f"Correct:    {'✓' if ex['correct'] else '✗'}\n\n")

print(f"Results saved to {results_file}")


Generating examples with model trained on 0.0% French...
nl: de apen eet de muizen → de apen eet de muizen gegeten paarden paarden paarden paarden jongens jongens jongens dokters dokters dokters dokters dokters dokters dokters | Target: de apen heeft de muizen gegeten
fr: le homme embrasse le fille → le homme embrasse le fille gemogen | Target: le homme a embrassé le fille
nl: de leeuwen slaat de katten → de leeuwen slaat de katten geslagen paarden paarden jongens jongens jongens jongens dokters dokters dokters vrouwen vrouwen vrouwen vrouwen vrouwen | Target: de leeuwen heeft de katten geslagen
fr: le docteur sauve le souris → le docteur sauve le souris jongen | Target: le docteur a sauvé le souris
fr: le docteur frappe le souris → le docteur frappe le souris de paarden paarden paarden jongens jongens meisjes meisjes meisjes meisjes vrouwen vrouwen vrouwen dokters dokters | Target: le docteur a frappé le souris
fr: le renard regarde le professeur → le renard regarde le professeur aap 

## small model

In [ ]:
import pandas as pd, numpy as np, torch, json
from transformers import GPT2Config, GPT2LMHeadModel, Trainer, TrainingArguments, PreTrainedTokenizerFast
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.decoders import WordPiece
from pathlib import Path

# Load small data sample
DATA_DIR = Path("data")
df = pd.read_csv(DATA_DIR/"train.csv").sample(n=100, random_state=0)  # smaller sample
vocab = json.load(open(DATA_DIR/"vocab.json"))
stoi = {t: i for i, t in enumerate(vocab)}

# Setup tokenizer
tok = Tokenizer(WordLevel(vocab=stoi, unk_token="<unk>"))
tok.pre_tokenizer = Whitespace()
tok.decoder = WordPiece()
tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tok, bos_token="<sos>", eos_token="<eos>",
    unk_token="<unk>", pad_token="<pad>"
)


In [ ]:
class HFDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.enc = tokenizer(df.input.tolist(), text_target=df.target.tolist(),
                             padding="max_length", truncation=True, max_length=20)
    def __len__(self): return len(self.enc["input_ids"])
    def __getitem__(self, i): return {k: torch.tensor(v[i]) for k, v in self.enc.items()}

ds = HFDataset(df)

model = GPT2LMHeadModel(GPT2Config(
    vocab_size=len(vocab), n_embd=128, n_layer=2, n_head=2,  # small model
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id
))


In [ ]:
def token_accuracy(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    mask = labels != tokenizer.pad_token_id
    acc = ((preds == labels) & mask).sum() / mask.sum()
    return {"token_accuracy": float(acc)}

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="tmp_check",
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        logging_steps=10,
        evaluation_strategy="steps", eval_steps=10,
        max_steps=100,
        save_strategy="no",
        report_to="none"
    ),
    train_dataset=ds,
    eval_dataset=ds,
    compute_metrics=token_accuracy
)

trainer.train()


## second method

In [10]:
# ==== Cell 1 : tiny debug dataset – robust concatenation =========================
import json, pandas as pd, torch, numpy as np
from pathlib import Path
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.decoders import WordPiece
from transformers import PreTrainedTokenizerFast

DATA_DIR = Path("data")
train_df = pd.read_csv(DATA_DIR/"train.csv")#.sample(200, random_state=0)
test_df  = pd.read_csv(DATA_DIR/"test.csv")#.sample(40 , random_state=0)

# --- tokenizer -----------------------------------------------------------------
vocab  = json.load(open(DATA_DIR/"vocab.json"))
special = ["<pad>","<sos>","<eos>","<sep>","<unk>"]
for tok in special:
    if tok not in vocab:
        vocab.append(tok)
stoi = {t:i for i,t in enumerate(vocab)}

tok_obj = Tokenizer(WordLevel(vocab=stoi, unk_token="<unk>"))
tok_obj.pre_tokenizer = Whitespace()
tok_obj.decoder       = WordPiece()
tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tok_obj,
    bos_token="<sos>", eos_token="<eos>",
    pad_token="<pad>", unk_token="<unk>",
    sep_token="<sep>"
)

MAXLEN = 14   # prompt + target + specials must fit

def encode_pair(present: str, past: str):
    pres_ids = tokenizer.encode(present, add_special_tokens=False)  # <-- fixed
    past_ids = tokenizer.encode(past   , add_special_tokens=False)  # <-- fixed

    margin = 3                  # <sos> <sep> <eos>
    max_prompt = MAXLEN - len(past_ids) - margin
    if max_prompt <= 0:
        raise RuntimeError("target sentence alone exceeds MAXLEN")
    pres_ids = pres_ids[:max_prompt]

    sos, sep, eos, pad = (
        tokenizer.bos_token_id,
        tokenizer.sep_token_id,
        tokenizer.eos_token_id,
        tokenizer.pad_token_id,
    )

    input_ids = [sos] + pres_ids + [sep] + past_ids + [eos]
    input_ids += [pad] * (MAXLEN - len(input_ids))

    sep_pos = 1 + len(pres_ids)          # index of <sep>
    labels  = [-100]*(sep_pos+1) + input_ids[sep_pos+1:]

    return {
        "input_ids": torch.tensor(input_ids),
        "attention_mask": torch.tensor([1 if t!=pad else 0 for t in input_ids]),
        "labels": torch.tensor(labels),
    }


class PairDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        enc = [encode_pair(r.input, r.target) for r in df.itertuples()]
        self.buf = {k: torch.stack([e[k] for e in enc]) for k in enc[0]}
    def __len__(self):  return len(self.buf["input_ids"])
    def __getitem__(self, i): return {k: v[i] for k, v in self.buf.items()}

train_ds = PairDataset(train_df)
val_ds   = PairDataset(test_df)

print("✓ dataset built")
print("Decoded example:")
print(tokenizer.decode(train_ds[0]['input_ids']))

✓ dataset built
Decoded example:
<sos> le professeur ignore le tigre <sep> le professeur a ignoré le tigre <eos>


In [19]:
print(test_df)

                               input                                  target  \
0              le chien suit le loup                le chien a suivi le loup   
1        les chiens frappe les loups           les chiens a frappé les loups   
2        les chiens chasse les loups           les chiens a chassé les loups   
3             le chien sauve le loup                le chien a sauvé le loup   
4              le chien voit le chat                   le chien a vu le chat   
...                              ...                                     ...   
5193  de vijanden draagt de vrienden  de vijanden heeft de vrienden gedragen   
5194    de vijand verzorgt de vriend      de vijand heeft de vriend verzorgd   
5195        de vijand redt de vriend         de vijand heeft de vriend gered   
5196    de vijanden redt de vrienden     de vijanden heeft de vrienden gered   
5197  de vijanden troost de vrienden  de vijanden heeft de vrienden getroost   

     lang  plural  
0      fr   False  

In [14]:
train_df.shape


(20794, 4)

In [15]:
lens = (train_df['input'] + " " + train_df['target']).apply(
        lambda s: len(tokenizer.tokenize(s)))
print(lens.describe())

count    20794.0
mean        11.0
std          0.0
min         11.0
25%         11.0
50%         11.0
75%         11.0
max         11.0
dtype: float64


In [16]:
# is cuda available?
print("CUDA available?", torch.cuda.is_available())


CUDA available? True


In [ ]:
# ==== Cell 2 : train tiny GPT‑2 for 1 000 steps =======================================
from transformers import GPT2Config, GPT2LMHeadModel, Trainer, TrainingArguments, set_seed
import random, numpy as np, torch

set_seed(42)
model = GPT2LMHeadModel(GPT2Config(
        vocab_size=len(vocab),
        n_embd=128, n_layer=2, n_head=2,
        bos_token_id=tokenizer.bos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id
))

def tok_acc(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    mask  = labels != -100
    return {"tok_acc": float(((preds==labels) & mask).sum() / mask.sum())}

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="debug_out",
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        max_steps=15_000,
        eval_steps=1000,
        logging_steps=500,
        learning_rate=5e-4,
        weight_decay=0.0,
        report_to="none",
        save_strategy="no"
    ),
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=tok_acc
)

trainer.train()

Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Step,Training Loss
500,0.724400
1000,0.027100
1500,0.017400
2000,0.013100
2500,0.006200
3000,0.002500
3500,0.003300
4000,0.001100
4500,0.001200
5000,0.003200


AttributeError: 'str' object has no attribute 'input'

In [22]:

# ---- quick qualitative check ---------------------------------------------------------
model.eval()
for row in test_df.sample(5).itertuples():
    prompt = f"<sos> {row.input} <sep>"
    ids = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**ids, max_new_tokens=20,
                         eos_token_id=tokenizer.eos_token_id, do_sample=False)
    raw = tokenizer.decode(out[0], skip_special_tokens=False)
    prediction = raw.split("<sep>")[1].replace("<eos>", "").strip()
    print(f"{row.input:45s}  →  {prediction:35s} | gold: {row.target}")


# Get the evaluation results
eval_results = trainer.evaluate()
print(f"Final evaluation accuracy: {eval_results['eval_tok_acc']:.4f}")


le oiseau ignore le souris                     →  le oiseau a ignoré le souris        | gold: le oiseau a ignoré le souris
le chien réconforte le enfant                  →  le chien a réconforté le enfant     | gold: le chien a réconforté le enfant
de paarden ziet de leeuwen                     →  de paarden heeft de leeuwen gezien  | gold: de paarden heeft de leeuwen gezien
le docteur frappe le enfant                    →  le docteur a frappé le enfant       | gold: le docteur a frappé le enfant
de man volgt de hond                           →  de man heeft de hond gevolgd        | gold: de man heeft de hond gevolgd


Final evaluation accuracy: 0.0000


In [23]:
model.eval()
correct = 0
total = 0

for row in test_df.itertuples():
    prompt = f"<sos> {row.input} <sep>"
    ids = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=20,
                            eos_token_id=tokenizer.eos_token_id, do_sample=False)
    
    raw = tokenizer.decode(out[0], skip_special_tokens=False)
    prediction = raw.split("<sep>")[1].replace("<eos>", "").strip()
    
    # Check if prediction matches target
    is_correct = prediction.strip() == row.target.strip()
    correct += int(is_correct)
    total += 1
    
    # Print with correctness indicator
    print(f"{row.input:45s} → {prediction:35s} | gold: {row.target} {'✓' if is_correct else '✗'}")

# Calculate and print accuracy
accuracy = correct / total
print(f"\nAccuracy on test set: {correct}/{total} = {accuracy:.4f} ({accuracy*100:.2f}%)")

le chien suit le loup                         → le chien a suivi le loup            | gold: le chien a suivi le loup ✓
les chiens frappe les loups                   → les chiens a frappé les les         | gold: les chiens a frappé les loups ✗
les chiens chasse les loups                   → les chiens a chassé les les         | gold: les chiens a chassé les loups ✗
le chien sauve le loup                        → le chien a sauvé le loup            | gold: le chien a sauvé le loup ✓
le chien voit le chat                         → le chien a vu le chat               | gold: le chien a vu le chat ✓
les chiens aime les chats                     → les chiens a aimé les chats         | gold: les chiens a aimé les chats ✓
le chien porte le chat                        → le chien a porté le chat            | gold: le chien a porté le chat ✓
les chiens porte les chats                    → les chiens a porté les gezien       | gold: les chiens a porté les chats ✗
les chiens embrasse les chats     

In [ ]:
#!/usr/bin/env python
# ---------------------------------------------------------------------------
# Tiny GPT‑2 (2‑layer, 128‑d) tense‑conversion trainer
# Usage: python train_tense_small.py --prop 0.3
# ---------------------------------------------------------------------------
import argparse, json, os, pandas as pd, numpy as np, torch, csv
from pathlib import Path
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.decoders import WordPiece
from transformers import (PreTrainedTokenizerFast,
                          GPT2Config, GPT2LMHeadModel,
                          Trainer, TrainingArguments,
                          EarlyStoppingCallback,
                          set_seed, default_data_collator)

# # ---------------------- CLI -------------------------------------------------
# parser = argparse.ArgumentParser()
# parser.add_argument("--prop", type=float, required=True,
#                     help="Proportion of French sentences in training mix (0–1).")
# args = parser.parse_args()
# p = args.prop
# assert 0.0 <= p <= 1.0
p=0.5
# ---------------------- seed & paths ---------------------------------------
set_seed(0)
DATA_DIR   = Path("data")
OUT_DIR    = Path(f"weights/small_p{int(p*100):02d}")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------- load data ------------------------------------------
train_full = pd.read_csv(DATA_DIR/"train.csv")
test_df    = pd.read_csv(DATA_DIR/"test.csv")

# balanced train split
df_fr = train_full[train_full.lang == "fr"]
df_nl = train_full[train_full.lang == "nl"]

# target counts under desired proportion p
want_fr = int(len(train_full) * p)
want_nl = len(train_full) - want_fr

# if either pool is too small, down‑sample the larger pool
if want_fr > len(df_fr):
    want_fr = len(df_fr)
    want_nl = int(want_fr * (1 - p) / p) if p > 0 else 0
elif want_nl > len(df_nl):
    want_nl = len(df_nl)
    want_fr = int(want_nl * p / (1 - p)) if p < 1 else 0

train_df = pd.concat([
    df_fr.sample(want_fr, replace=False, random_state=0),
    df_nl.sample(want_nl, replace=False, random_state=0)
]).sample(frac=1, random_state=0).reset_index(drop=True)

print(f"Train set: {len(train_df)} rows "
      f"({len(train_df[train_df.lang=='fr'])} FR, "
      f"{len(train_df[train_df.lang=='nl'])} NL)")

# ---------------------- tokenizer ------------------------------------------
vocab = json.load(open(DATA_DIR/"vocab.json"))
for tok in ["<pad>","<sos>","<eos>","<sep>","<unk>"]:
    if tok not in vocab: vocab.append(tok)
stoi = {t:i for i,t in enumerate(vocab)}

tok_obj = Tokenizer(WordLevel(vocab=stoi, unk_token="<unk>"))
tok_obj.pre_tokenizer = Whitespace()
tok_obj.decoder       = WordPiece()
tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tok_obj,
    bos_token="<sos>", eos_token="<eos>",
    pad_token="<pad>", unk_token="<unk>",
    sep_token="<sep>"
)

# ---------------------- encode function ------------------------------------
def encode_pair(present, past):
    pres_ids = tokenizer.encode(present, add_special_tokens=False)
    past_ids = tokenizer.encode(past   , add_special_tokens=False)
    ids = (
        [tokenizer.bos_token_id] +
        pres_ids +
        [tokenizer.sep_token_id] +
        past_ids +
        [tokenizer.eos_token_id]
    )
    sep_pos = 1 + len(pres_ids)
    labels  = [-100]*(sep_pos+1) + ids[sep_pos+1:]
    return {"input_ids": ids,
            "attention_mask": [1]*len(ids),
            "labels": labels}

class PairDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        enc = [encode_pair(r.input, r.target) for r in df.itertuples()]
        self.data = enc
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        return {k: torch.tensor(v) for k,v in self.data[i].items()}

train_ds = PairDataset(train_df)
val_ds   = PairDataset(test_df)

# ---------------------- model ---------------------------------------------
model = GPT2LMHeadModel(GPT2Config(
        vocab_size=len(vocab), n_embd=128, n_layer=2, n_head=2,
        bos_token_id=tokenizer.bos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id
)).to("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------- metrics -------------------------------------------
def tok_acc(eval_pred):
    logits, labels = eval_pred          # numpy arrays
    # shift logits to align with labels
    preds = np.argmax(logits[:, :-1], axis=-1)   # drop last time‑step
    lbls  = labels[:, 1:]                         # drop first token
    mask  = lbls != -100                         # same ignore index
    correct = (preds == lbls) & mask
    return {"tok_acc": float(correct.sum() / mask.sum())}

def exact_match(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits[:, :-1], axis=-1)
    lbls  = labels[:, 1:]
    mask  = lbls != -100
    # a sequence is correct if *all* masked positions match
    seq_ok = ((preds == lbls) | ~mask).all(axis=-1)
    return {"exact_match": float(seq_ok.mean())}


def compute_metrics(eval_pred):
    out = tok_acc(eval_pred)
    out.update(exact_match(eval_pred))
    return out

# ---------------------- training args --------------------------------------
args_t = TrainingArguments(
    output_dir=str(OUT_DIR),
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,      # effective 64
    fp16=True,
    learning_rate=2e-4,
    warmup_steps=500,
    max_steps=20000,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    eval_strategy="steps", eval_steps=1000,
    save_strategy="steps", save_steps=1000, save_total_limit=5,
    logging_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="tok_acc",
    greater_is_better=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args_t,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=default_data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

# ---------------------- train ------------------------------------------------
trainer.train()
trainer.save_model(OUT_DIR/"final")
tokenizer.save_pretrained(OUT_DIR/"final")

# ---------------------- evaluate on full test set ---------------------------
model.eval()
gens = []
device = model.device

for r in test_df.itertuples():
    prompt = f"<sos> {r.input} <sep>"
    ids = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**ids,
                             max_new_tokens=20,
                             eos_token_id=tokenizer.eos_token_id,
                             do_sample=False,
                             num_beams=4)
    raw = tokenizer.decode(out[0], skip_special_tokens=False)
    pred = raw.split("<sep>")[1].replace("<eos>", "").strip()
    gens.append({"input": r.input, "target": r.target,
                 "prediction": pred, "lang": r.lang})

pd.DataFrame(gens).to_csv(OUT_DIR/"test_generations.csv", index=False)

# ---------------------- save metrics ---------------------------------------
metrics = {
    "train_final_step": trainer.state.global_step,
    **trainer.state.log_history[-1]   # last logged metrics
}
with open(OUT_DIR/"metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(f"✓ finished run p={p:.1f}  |  outputs saved in {OUT_DIR}")


Train set: 20770 rows (10385 FR, 10385 NL)


/n/home06/drooryck/circuits_languages_2/venv39/lib/python3.10/site-packages/accelerate/accelerator.py:449: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Step,Training Loss,Validation Loss,Tok Acc,Exact Match
1000,0.032600,0.014308,0.142857,0.000000
2000,0.006000,0.002711,0.142857,0.000000
3000,0.003400,0.001208,0.142857,0.000000
4000,0.001500,0.000564,0.142280,0.000000
5000,0.000900,0.000275,0.123454,0.000000
6000,0.000600,0.000165,0.133101,0.000000


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


✓ finished run p=0.5  |  outputs saved in weights/small_p50
